## Orchestrating Parallel Agent Systems

# Unit 26: High-Density Orchestration Topologies and Asynchronous Tool Delegation

## Introduction & Overview

Welcome back! In the previous modules, you constructed a fully asynchronous multi-agent orchestration engine capable of managing isolated conversation states concurrently and executing deterministic tools in parallel. Now, we are ready to take this parallelization to its ultimate architectural state: building an **orchestrator coordinator agent** that can programmatically delegate work to downstream specialized agents wrapped as tools.

In this lesson, you will analyze two distinct design methodologies for wrapping autonomous agents as tools: the **Synchronous Wrapper Approach** and the **Asynchronous Wrapper Approach**. You will evaluate the systemic trade-offs of each pattern and re-engineer your core `Agent` class runtime handler to detect and route both synchronous and asynchronous tools seamlessly.

---

## Understanding Agent Orchestration Topologies

**Agent Orchestration** is an architectural pattern in which a single central coordinator agent manages, schedules, and delegates tasks to multiple down-stream specialized agents. Think of it like an engineering project manager who receives a complex technical requirements document, breaks it down into decoupled functional subproblems, and assigns each component to a separate engineer with domain-specific expertise. The coordinator doesn't need to know how to resolve every technical detail directly; it just needs to understand the problem space well enough to parse and delegate the payload effectively.

When you pass a compound problem to this orchestrator—such as calculating multi-tier checkout invoices from three separate digital store branches—it cross-references the prompt layout, recognizes that each branch calculation is completely independent, and fires them **in parallel**. Instead of executing the evaluation loops sequentially, the orchestrator targets each problem by spinning up separate specialized agent calls concurrently.

This model blends the optimization bounds of **functional specialization** (keeping agents small, focused, and accurate) with **asynchronous concurrency**, ensuring that adding new specialized sub-agents or handling heavier nested datasets requires zero structural modifications to your core loop controllers.

---

## Building the Downstream Specialist Agent

Before standing up our central coordinator, we must instantiate the target specialist agent that will accept the delegated calculation batches. This calculator agent is packed with all standard math primitives, making it a highly reliable standalone resource for resolving numeric operations:

```python
import json
from agent import Agent
from functions import sum_numbers, multiply_numbers, subtract_numbers, divide_numbers, power, square_root

# Load technical schema manifests from disk
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Build internal math tool registry lookup
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Instantiate the specialized worker agent
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt=(
        "You are a specialized calculator assistant. You focus entirely on resolving "
        "complex mathematical calculations, equations, and algorithmic formulas."
    ),
    tools=math_tools,
    tool_schemas=tool_schemas
)

```

The `calculator_assistant` has a singular focus. Its system prompt strictly isolates its operational boundaries, and it remains completely unaware of parent orchestration mechanics, list fragment allocations, or adjacent task fanning. It simply ingests an inbound context string, executes its local math tools, and returns a clean answer block.

---

## Design Pattern 1: Synchronous Agent Wrappers

The most direct way to transform an agent instance into an executable tool is to build a standard synchronous closure function. This wrapper abstracts the asynchronous initialization details away from the tool caller:

```python
def create_agent_tool_sync(agent: Agent, description: str) -> tuple:
    """
    Constructs a traditional synchronous tool function wrapping an Agent instance.
    This function blocks the executing thread, meaning it must be offloaded to a worker pool.
    """
    def agent_tool_function(message: str) -> str:
        # Blocks the executing thread and instantiates a temporary nested loop lifecycle
        _, response = agent.run_sync([{"role": "user", "content": message}])
        print(f"📊 [Thread Sync Bridge] Sub-Agent '{agent.name}' tool pipeline closed.")
        return response

    # Standard tool registration schema structure mapping
    tool_schema = {
        "name": f"{agent.name}_agent",
        "description": description,
        "input_schema": {
            "type": "object",
            "properties": {
                "message": {
                    "type": "string",
                    "description": "The precise sub-problem context or message packet to send to the sub-agent."
                }
            },
            "required": ["message"]
        }
    }
    return agent_tool_function, tool_schema

```

### Architectural Trade-Off Analysis:

* **The Underlying Worker Thread Mechanics:** When the orchestrator invokes this tool via a standard loop, the runtime maps the function name to `agent_tool_function`. Because it is detected as a regular function, it is executed via `asyncio.to_thread()`, spinning up an OS-level worker thread. Inside that thread, `run_sync()` spins up a **nested, isolated event loop** to run the agent's core `AsyncAnthropic` connection.
* **System Bottlenecks:** While this pattern is simple and requires zero changes to your existing code, it introduces noticeable resource overhead at scale. Running three or four agents in parallel requires allocating three or four separate native threads and matching event loops. This consumes far more memory and CPU cycles than driving everything inside a single, unified execution thread.

---

## Design Pattern 2: Asynchronous Agent Wrappers

To build production-grade agent networks that scale efficiently under heavy enterprise loads, you should implement **True Asynchronous Tool Wrapper Closures**. This pattern allows sub-agents to run directly as concurrent tasks inside the parent event loop without spinning up extra worker threads:

```python
def create_agent_tool_async(agent: Agent, description: str) -> tuple:
    """
    Constructs a native non-blocking asynchronous tool function wrapping an Agent instance.
    Runs directly on the primary system event loop thread.
    """
    async def agent_tool_function(message: str) -> str:
        # Clean coroutine invocation passing context down the primary event loop thread
        _, response = await agent.run([{"role": "user", "content": message}])
        print(f"📊 [Event Loop Task] Sub-Agent '{agent.name}' tool pipeline closed.")
        return response

    tool_schema = {
        "name": f"{agent.name}_agent",
        "description": description,
        "input_schema": {
            "type": "object",
            "properties": {
                "message": {
                    "type": "string",
                    "description": "The precise sub-problem context or message packet to send to the sub-agent."
                }
            },
            "required": ["message"]
        }
    }
    return agent_tool_function, tool_schema

```

### Critical Implementation Obstacle:

Look closely at the declaration statement: `async def agent_tool_function(message)`. This function returns an asynchronous coroutine object. If we pass this tool function directly into our previous `call_tool()` engine implementation, **the execution will crash**.

Our older loop assumes all tool functions are legacy synchronous blocks and automatically routes them through `asyncio.to_thread()`. However, `to_thread()` expects a standard function pointer; passing it a coroutine function will trigger a runtime typing failure. To support true async tools, we must teach our parent `Agent` class how to dynamically inspect and handle different tool types at runtime.

---

## Dynamic Function Profiling via the `inspect` Module

To make our agent orchestrator intelligent enough to handle both synchronous legacy code and native async tool closures, we can use Python's built-in `inspect.iscoroutinefunction()` utility. This profiles function signatures dynamically to determine their underlying execution requirements:

```python
import inspect
import asyncio

def standard_tool():
    return "Blocking calculation complete."

async def async_tool():
    return "Non-blocking coroutine calculation complete."

# Evaluating execution footprints
print(inspect.iscoroutinefunction(standard_tool))  # Output: False
print(inspect.iscoroutinefunction(async_tool))   # Output: True

```

By adding this inspection check to our core orchestration logic, we can accurately route incoming tools to their optimal execution path based on how they were declared.

---

## Implementing the Intelligent Tool Routing Matrix

Let's re-engineer the core `call_tool()` method inside your `Agent` class definition to build a multi-route tool execution handler. This allows the system to seamlessly switch between event loop execution and background thread pooling:

```python
# Updated method addition to the core Agent class in agent.py
import inspect
import asyncio

async def call_tool(self, tool_use) -> dict:
    """
    Dynamically analyzes the execution signature of target tools to route them 
    either directly onto the event loop thread or safely offload them to background pools.
    """
    tool_name = tool_use.name
    tool_input = tool_use.input or {}
    tool_use_id = tool_use.id
    
    print(f"🔧 Invoking Registered Tool Target: {tool_name}({tool_input})")
    
    try:
        tool_fn = self.tools[tool_name]
    except KeyError:
        result = f"Error: The requested tool token '{tool_name}' is missing from the registry map."
    except Exception as e:
        result = f"Error during tool lookup phase: {str(e)}"
    else:
        try:
            # Step 1: Query the signature space using reflection
            if inspect.iscoroutinefunction(tool_fn):
                print(f"⚡ [Event Loop Route] Direct coroutine execution path allocated for '{tool_name}'.")
                # Route A: Run native async tools directly on the primary event loop thread
                result_value = await tool_fn(**tool_input)
            else:
                print(f"🧵 [Thread Pool Route] Offloading legacy synchronous function '{tool_name}' to worker pool.")
                # Route B: Offload blocking synchronous tools to an isolated worker thread pool
                result_value = await asyncio.to_thread(tool_fn, **tool_input)
                
            result = str(result_value)
        except Exception as e:
            result = f"Runtime error during internal execution of '{tool_name}': {str(e)}"
            
    return {
        "type": "tool_result",
        "tool_use_id": tool_use_id,
        "content": result
    }

```

---

## Architectural Comparison: Sync vs. Async Tool Wrappers

| Capability / Attribute | Synchronous Agent Tool Wrapper (`create_agent_tool_sync`) | Asynchronous Agent Tool Wrapper (`create_agent_tool_async`) |
| --- | --- | --- |
| **Call Execution Signature** | Returns standard blocking function strings. | Returns a native non-blocking coroutine. |
| **Event Loop Topology** | Allocates an independent nested loop structure for every distinct call. | Operates directly within the context of the main event loop. |
| **System Thread Overhead** | Stacks thread allocations linearly ($\mathcal{O}(N)$ OS thread footprint). | Uses zero extra threads; runs entirely on the primary event loop thread ($\mathcal{O}(1)$). |
| **Scale Constraints** | Scales poorly; subject to thread context switching overhead under heavy concurrent load. | Scales exceptionally well; capable of fanning out hundreds of concurrent sub-agent calls. |
| **Refactoring Requirements** | Drop-in ready; compatible with basic legacy agent execution loops. | Requires runtime signature inspection hooks (`inspect.iscoroutinefunction`). |

---

## Instantiating the Orchestrator with Async Tools

Now that our agent class includes intelligent tool routing, let's assemble our master orchestrator. We will wrap our downstream math specialist as an async tool, allowing the coordinator to delegate tasks across the event loop:

```python
# Wrap the downstream calculator assistant as an async tool coroutine
calc_tool_fn, calc_tool_schema = create_agent_tool_async(
    calculator_assistant,
    description="Dispatches a refined mathematical prompt to a specialized sub-agent calculator."
)

# Instantiate the Master Orchestrator Coordinator
orchestrator_agent = Agent(
    name="orchestrator_agent",
    system_prompt=(
        "You are an executive multi-agent math orchestration coordinator. When given complex, "
        "multi-step analytical word problems, you MUST analyze the intent, break down the core logic "
        "into independent subproblems, and delegate each subproblem to an individual calculator tool call. "
        "Always issue your calculator_assistant_agent calls IN PARALLEL within the same conversation turn "
        "whenever subproblems are independent. Do not calculate values yourself. "
        "After all sub-agents return their text responses, synthesize the data into a comprehensive final answer."
    ),
    tools={calc_tool_schema["name"]: calc_tool_fn},
    tool_schemas=[calc_tool_schema]
)

```

---

## Executing the Orchestrator on Complex Datasets

Let's test our complete parallel orchestration system with a multi-layered finance prompt that requires evaluating three independent data arrays simultaneously:

```python
import asyncio

async def run_orchestration_matrix():
    messages = [
        {
            'role': 'user',
            'content': (
                'I need to calculate the total cost of three different purchases:\n'
                '- Store A: 15 items at $12.50 each, plus $8.99 shipping\n'
                '- Store B: 23 items at $9.75 each, plus $12.50 shipping\n'
                '- Store C: 8 items at $24.99 each, plus $6.25 shipping\n'
                'What is the grand total across all three stores?'
            )
        }
    ]
    
    print("🚀 Triggering Master Orchestration Fan-out Sequence...")
    _, grand_summary = await orchestrator_agent.run(messages)
    
    print('\n=== Final Synthesized System Response ===\n')
    print(grand_summary)

if __name__ == "__main__":
    asyncio.run(run_orchestration_matrix())

```

---

## Deep Diagnostic Trace Analysis

When you run this architecture, the stdout execution logs show a clear trace of parallel orchestration and dynamic tool routing in action:

```text
🚀 Triggering Master Orchestration Fan-out Sequence...

🔧 Invoking Registered Tool Target: calculator_assistant_agent({'message': 'Calculate Store A total cost: 15 items at $12.50 each + $8.99 shipping'})
⚡ [Event Loop Route] Direct coroutine execution path allocated for 'calculator_assistant_agent'.

🔧 Invoking Registered Tool Target: calculator_assistant_agent({'message': 'Calculate Store B total cost: 23 items at $9.75 each + $12.50 shipping'})
⚡ [Event Loop Route] Direct coroutine execution path allocated for 'calculator_assistant_agent'.

🔧 Invoking Registered Tool Target: calculator_assistant_agent({'message': 'Calculate Store C total cost: 8 items at $24.99 each + $6.25 shipping'})
⚡ [Event Loop Route] Direct coroutine execution path allocated for 'calculator_assistant_agent'.

   🔧 [Agent Context: calculator_assistant] Tool called: multiply_numbers({'a': 8, 'b': 24.99})
   🧵 [Thread Pool Route] Offloading legacy synchronous function 'multiply_numbers' to worker pool.
   🔧 [Agent Context: calculator_assistant] Tool called: multiply_numbers({'a': 23, 'b': 9.75})
   🧵 [Thread Pool Route] Offloading legacy synchronous function 'multiply_numbers' to worker pool.
   🔧 [Agent Context: calculator_assistant] Tool called: multiply_numbers({'a': 15, 'b': 12.5})
   🧵 [Thread Pool Route] Offloading legacy synchronous function 'multiply_numbers' to worker pool.

   🔧 [Agent Context: calculator_assistant] Tool called: sum_numbers({'a': 187.5, 'b': 8.99})
   🧵 [Thread Pool Route] Offloading legacy synchronous function 'sum_numbers' to worker pool.
   ...
📊 [Event Loop Task] Sub-Agent 'calculator_assistant' tool pipeline closed.
📊 [Event Loop Task] Sub-Agent 'calculator_assistant' tool pipeline closed.
📊 [Event Loop Task] Sub-Agent 'calculator_assistant' tool pipeline closed.

```

### Key Technical Takeaways from the Trace:

1. **Parallel Task Fan-out:** The orchestrator instantly generates three concurrent `calculator_assistant_agent` task calls within a single conversation turn.
2. **Zero Thread Overhead at the Coordinator Level:** Because the sub-agents are wrapped in async closures, the core tool engine routes them straight through the `⚡ [Event Loop Route]` branch. They run side-by-side as non-blocking tasks on the main thread, without spinning up nested loops or extra OS threads.
3. **Hybrid Execution Environment:** Once inside the `calculator_assistant` sub-agent instances, the mathematical operations are identified as synchronous functions (e.g., `multiply_numbers`). The handler dynamically switches gears and routes them through the thread-safe `🧵 [Thread Pool Route]` branch using `asyncio.to_thread()`.
4. **Complete Interleaved Concurrency:** The math tools from all three calculator sub-agents execute at roughly the same time, showing true parallel execution across both the orchestration and mathematical calculation levels.

---

## Examining the Final Synthesized Response

Once all three calculator sub-agents complete their concurrent operations and return their data blocks, the master orchestrator collects the outputs, merges the context metrics, and formats a clean final summary:

```markdown
=== Final Synthesized System Response ===

## Total Cost Breakdown

Here's the complete breakdown of your purchases across all three stores:

**Store A:** $196.49
- 15 items × $12.50 = $187.50
- Shipping: $8.99

**Store B:** $236.75
- 23 items × $9.75 = $224.25  
- Shipping: $12.50

**Store C:** $206.17
- 8 items × $24.99 = $199.92
- Shipping: $6.25

**Grand Total: $639.41**

```

The orchestrator successfully parsed the user's complex query, broken it down into standalone subproblems, delegated the operations to parallel downstream agents, and combined the raw data points back into a clean, context-aware summary.

---

## Summary & Core Milestones

Congratulations, you have completed the final lesson of this advanced orchestration course! Let's reflect on the production-grade agent architecture you have built step-by-step:

* **The Core Agent Foundations:** You built an autonomous class engine capable of parsing Claude's structured response tokens and maintaining conversational states across multiple turns.
* **Asynchronous Integration:** You moved your network architecture to the non-blocking `AsyncAnthropic` client core, allowing single-threaded systems to handle multiple active user streams concurrently.
* **Parallel Tool Execution:** You added concurrent execution layers that group standard tool operations into batch arrays using `asyncio.create_task()` and run them simultaneously via `asyncio.gather()`.
* **Hierarchical Agent Topologies:** You used runtime function reflection (`inspect.iscoroutinefunction`) to build a hybrid tool engine that seamlessly coordinates both synchronous code utilities and asynchronous sub-agents in parallel.

These architectural design patterns form the core framework used by top-tier AI engineering teams to build scalable, production-grade enterprise multi-agent networks. You are now fully prepared to head into the final practice labs, complete your engineering exercises, and deploy high-performance agent meshes! Happy coding! 🚀

## Completing the Parallel Agent Orchestrator

It's time to bring the orchestrator pattern to life by completing the critical pieces that enable parallel agent delegation!

In this exercise, you'll:

    Finish implementing the create_agent_tool() function by calling the agent's run_sync() method with the correct message format. Pass a list containing a dictionary with "role" and "content" keys, where the content comes from the message parameter.

    Run the orchestrator agent using its async run() method with the provided messages list, and unpack both the result messages and the response. Use await here because you're in an async context and want to leverage async/await for proper concurrency.

    Print the orchestrator's final response to display the synthesized answer after coordinating multiple calculator agents working in parallel.

Once complete, run your code to see the orchestrator coordinate multiple calculator agents in parallel, breaking down the three-store problem and synthesizing their results into a complete answer!

```
# main.py
import asyncio
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}


def create_agent_tool(agent, description):
    """
    Create a tool function and schema for using an agent as a tool.

    Args:
        agent: The agent to wrap as a tool
        description: Description of what this agent tool does

    Returns:
        tuple: (tool_function, tool_schema)
    """
    def agent_tool_function(message):
        # TODO: Call agent.run_sync() and unpack the response (ignore the first return value)

        print(f"📊 Agent ({agent.name}) as tool finished working")

        return response

    # Create schema for this agent tool
    tool_schema = {
        "name": f"{agent.name}_agent",
            "description": description,
            "input_schema": {
                "type": "object",
                "properties": {
                    "message": {
                        "type": "string",
                        "description": "The message to send to the agent"
                    }
                },
                "required": ["message"]
            }
    }

    return agent_tool_function, tool_schema


async def main():
    # Create a calculator assistant
    calculator_assistant = Agent(
        name="calculator_assistant",
        system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
        tools=math_tools,
        tool_schemas=tool_schemas
    )

    # Create agent tool for calculator
    calculator_tool_function, calculator_tool_schema = create_agent_tool(
        calculator_assistant,
        "Call the calculator assistant to solve a single mathematical problem or equation."
    )

    # Create the orchestrator assistant
    orchestrator = Agent(
        name="orchestrator",
        system_prompt=(
            "You are a math orchestration coordinator. When given complex problems with multiple independent "
            "subproblems, you MUST break them down and delegate each subproblem to a separate calculator agent call. "
            "Issue multiple calculator_assistant_agent tool calls IN PARALLEL within the same turn for independent "
            "calculations. Each agent call should handle ONE specific calculation or equation. "
            "After all agents return results, synthesize them into a clear, complete final answer."
        ),
        tools={calculator_tool_schema["name"]: calculator_tool_function},
        tool_schemas=[calculator_tool_schema]
    )

    # Run the orchestrator on a problem that requires breaking into parallel subproblems
    messages = [
        {
            'role': 'user',
            'content': (
                'I need to calculate the total cost of three different purchases:\n'
                '- Store A: 15 items at $12.50 each, plus $8.99 shipping\n'
                '- Store B: 23 items at $9.75 each, plus $12.50 shipping\n'
                '- Store C: 8 items at $24.99 each, plus $6.25 shipping\n'
                'What is the grand total across all three stores?'
            )
        }
    ]
    
    # TODO: Run orchestrator.run(messages) and unpack result_messages and response
    
    print('\n=== Final Response ===\n')
    # TODO: Print the orchestrator's final response
    


if __name__ == "__main__":
    asyncio.run(main())



# agent.py
import asyncio
from anthropic import AsyncAnthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = AsyncAnthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)
        
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        all_tools = []
        
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    async def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            tool_fn = self.tools[tool_name]
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"
        else:
            try:
                result = str(await asyncio.to_thread(tool_fn, **tool_input))
            except Exception as e:
                result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
        
    async def call_handoff(self, tool_use, messages):
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            result = await target_agent.run(clean_messages)

            return True, result
        except StopIteration:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    async def run(self, input_messages):
        messages = input_messages.copy()

        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = await self.client.messages.create(
                **self._build_request_args(messages)
            )

            messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "tool_use":
                tool_results = []
                tasks = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        if content_item.name == "handoff":
                            handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                            if handoff_success:
                                return handoff_result
                            else:
                                tool_results.append(handoff_result)
                        else:
                            tasks.append(asyncio.create_task(self.call_tool(content_item)))

                if tasks:
                    tool_results.extend(await asyncio.gather(*tasks))

                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                response_text = self._extract_text(response.content)

                return messages, response_text

        raise Exception("Max turns reached")

    def run_sync(self, input_messages):
        return asyncio.run(self.run(input_messages))

    
```

Here is the complete and corrected implementation for your parallel orchestration task.

The `create_agent_tool` function is completed by packing the inbound tool parameter `message` inside a list-wrapped conversation dictionary, executing it using the sub-agent's `run_sync()` method, and capturing the plain text answer. The `main()` routine uses `await orchestrator.run(messages)` to execute the parent coordinator on the main thread loop.

### 1. Completed Orchestrator Entrypoint (`main.py`)

```python
# main.py
import asyncio
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}


def create_agent_tool(agent, description):
    """
    Create a tool function and schema for using an agent as a tool.

    Args:
        agent: The agent to wrap as a tool
        description: Description of what this agent tool does

    Returns:
        tuple: (tool_function, tool_schema)
    """
    def agent_tool_function(message):
        # Call agent.run_sync() with proper message formatting and unpack the response text
        _, response = agent.run_sync([{"role": "user", "content": message}])

        print(f"📊 Agent ({agent.name}) as tool finished working")

        return response

    # Create schema for this agent tool
    tool_schema = {
        "name": f"{agent.name}_agent",
        "description": description,
        "input_schema": {
            "type": "object",
            "properties": {
                "message": {
                    "type": "string",
                    "description": "The message to send to the agent"
                }
            },
            "required": ["message"]
        }
    }

    return agent_tool_function, tool_schema


async def main():
    # Create a calculator assistant
    calculator_assistant = Agent(
        name="calculator_assistant",
        system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
        tools=math_tools,
        tool_schemas=tool_schemas
    )

    # Create agent tool for calculator
    calculator_tool_function, calculator_tool_schema = create_agent_tool(
        calculator_assistant,
        "Call the calculator assistant to solve a single mathematical problem or equation."
    )

    # Create the orchestrator assistant
    orchestrator = Agent(
        name="orchestrator",
        system_prompt=(
            "You are a math orchestration coordinator. When given complex problems with multiple independent "
            "subproblems, you MUST break them down and delegate each subproblem to a separate calculator agent call. "
            "Issue multiple calculator_assistant_agent tool calls IN PARALLEL within the same turn for independent "
            "calculations. Each agent call should handle ONE specific calculation or equation. "
            "After all agents return results, synthesize them into a clear, complete final answer."
        ),
        tools={calculator_tool_schema["name"]: calculator_tool_function},
        tool_schemas=[calculator_tool_schema]
    )

    # Run the orchestrator on a problem that requires breaking into parallel subproblems
    messages = [
        {
            'role': 'user',
            'content': (
                'I need to calculate the total cost of three different purchases:\n'
                '- Store A: 15 items at $12.50 each, plus $8.99 shipping\n'
                '- Store B: 23 items at $9.75 each, plus $12.50 shipping\n'
                '- Store C: 8 items at $24.99 each, plus $6.25 shipping\n'
                'What is the grand total across all three stores?'
            )
        }
    ]
    
    print("🚀 Running orchestrator topology loop across the event loop context...")
    # Run orchestrator.run(messages) and unpack result_messages and response
    result_messages, response = await orchestrator.run(messages)
    
    print('\n=== Final Response ===\n')
    # Print the orchestrator's final response
    print(response)


if __name__ == "__main__":
    asyncio.run(main())

```

---

### 2. Core Agent Runtime Architecture (`agent.py`)

This file contains your foundational agent configuration framework. It handles async message pooling and includes the mandatory `run_sync` method exposed to the agent tool handler wrapper closure:

```python
# agent.py
import asyncio
from anthropic import AsyncAnthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = AsyncAnthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)
        
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        all_tools = []
        
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    async def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            tool_fn = self.tools[tool_name]
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"
        else:
            try:
                result = str(await asyncio.to_thread(tool_fn, **tool_input))
            except Exception as e:
                result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
        
    async def call_handoff(self, tool_use, messages):
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            result = await target_agent.run(clean_messages)

            return True, result
        except StopIteration:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    async def run(self, input_messages):
        messages = input_messages.copy()

        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = await self.client.messages.create(
                **self._build_request_args(messages)
            )

            messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "tool_use":
                tool_results = []
                tasks = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        if content_item.name == "handoff":
                            handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                            if handoff_success:
                                return handoff_result
                            else:
                                tool_results.append(handoff_result)
                        else:
                            tasks.append(asyncio.create_task(self.call_tool(content_item)))

                if tasks:
                    tool_results.extend(await asyncio.gather(*tasks))

                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                response_text = self._extract_text(response.content)

                return messages, response_text

        raise Exception("Max turns reached")

    def run_sync(self, input_messages):
        return asyncio.run(self.run(input_messages))

```

## Exploring Async vs Sync Agent Tools

Time to discover what happens when we try to make agent tools truly asynchronous!

In this exercise, you'll modify the create_agent_tool() function to use await agent.run() instead of agent.run_sync(). This seemingly natural change will reveal why mixing async functions with thread-based execution doesn't work as expected.

Your task is to update the agent_tool_function:

    Change its definition to be asynchronous.
    Replace the call to agent.run_sync() with await agent.run().

Once you make this change, run your code and check the Error tab. You'll see a RuntimeWarning: coroutine was never awaited message. This happens because asyncio.to_thread() tries to execute your async function in a worker thread—but threads can't await coroutines! This error teaches you why we use run_sync() inside tool functions: it provides a synchronous interface that works perfectly with thread-based parallel execution, even though we want concurrent agent calls. This is a crucial insight for building robust orchestration systems!

```
# agent.py


# main.py
import asyncio
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}


def create_agent_tool(agent, description):
    """
    Create a tool function and schema for using an agent as a tool.

    Args:
        agent: The agent to wrap as a tool
        description: Description of what this agent tool does

    Returns:
        tuple: (tool_function, tool_schema)
    """
    def agent_tool_function(message):
        # Call the agent synchronously inside the tool (executed in a worker thread)
        _, response = agent.run_sync([{"role": "user", "content": message}])

        print(f"📊 Agent ({agent.name}) as tool finished working")

        return response

    # Create schema for this agent tool
    tool_schema = {
        "name": f"{agent.name}_agent",
            "description": description,
            "input_schema": {
                "type": "object",
                "properties": {
                    "message": {
                        "type": "string",
                        "description": "The message to send to the agent"
                    }
                },
                "required": ["message"]
            }
    }

    return agent_tool_function, tool_schema


async def main():
    # Create a calculator assistant
    calculator_assistant = Agent(
        name="calculator_assistant",
        system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
        tools=math_tools,
        tool_schemas=tool_schemas
    )

    # Create agent tool for calculator
    calculator_tool_function, calculator_tool_schema = create_agent_tool(
        calculator_assistant,
        "Call the calculator assistant to solve a single mathematical problem or equation."
    )

    # Create the orchestrator assistant
    orchestrator = Agent(
        name="orchestrator",
        system_prompt=(
            "You are a math orchestration coordinator. When given complex problems with multiple independent "
            "subproblems, you MUST break them down and delegate each subproblem to a separate calculator agent call. "
            "Issue multiple calculator_assistant_agent tool calls IN PARALLEL within the same turn for independent "
            "calculations. Each agent call should handle ONE specific calculation or equation. "
            "After all agents return results, synthesize them into a clear, complete final answer."
        ),
        tools={calculator_tool_schema["name"]: calculator_tool_function},
        tool_schemas=[calculator_tool_schema]
    )

    # Run the orchestrator on a problem that requires breaking into parallel subproblems
    messages = [
        {
            'role': 'user',
            'content': (
                'I need to calculate the total cost of three different purchases:\n'
                '- Store A: 15 items at $12.50 each, plus $8.99 shipping\n'
                '- Store B: 23 items at $9.75 each, plus $12.50 shipping\n'
                '- Store C: 8 items at $24.99 each, plus $6.25 shipping\n'
                'What is the grand total across all three stores?'
            )
        }
    ]
    result_messages, response = await orchestrator.run(messages)

    print('\n=== Final Response ===\n')
    print(response)


if __name__ == "__main__":
    asyncio.run(main())


```

By shifting `agent_tool_function` to an asynchronous coroutine while still routing it through a thread-bound parallelizer, you expose a fundamental constraint of Python's concurrency landscape.

When your agent's internal loop triggers `asyncio.to_thread(tool_fn, tool_input)`, it expects `tool_fn` to be a plain, blocking synchronous function. If you pass it an `async def` function, `to_thread` calls it like a standard function pointer, which merely returns an un-awaited coroutine object instead of running the code. The thread exits instantly, the coroutine is dropped, and Python throws a `RuntimeWarning: coroutine was never awaited`.

Here is the updated `main.py` script containing the intentionally flawed asynchronous tool modification to help you observe this exact error behavior:

### Completed `main.py` with Intentional Coroutine Mismatch

```python
# main.py
import asyncio
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}


def create_agent_tool(agent, description):
    """
    Create a tool function and schema for using an agent as a tool.
    
    CRITICAL LESSON: This implementation will intentionally trigger an error 
    because our current Agent loop offloads this function to a worker thread via 
    asyncio.to_thread(), which cannot execute or await native coroutines!
    """
    # 1. Changed the inner function definition to be asynchronous
    async def agent_tool_function(message):
        # 2. Replaced the call to agent.run_sync() with 'await agent.run()'
        _, response = await agent.run([{"role": "user", "content": message}])

        print(f"📊 Agent ({agent.name}) as tool finished working")

        return response

    # Create schema for this agent tool
    tool_schema = {
        "name": f"{agent.name}_agent",
        "description": description,
        "input_schema": {
            "type": "object",
            "properties": {
                "message": {
                    "type": "string",
                    "description": "The message to send to the agent"
                }
            },
            "required": ["message"]
        }
    }

    return agent_tool_function, tool_schema


async def main():
    # Create a calculator assistant
    calculator_assistant = Agent(
        name="calculator_assistant",
        system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
        tools=math_tools,
        tool_schemas=tool_schemas
    )

    # Create agent tool for calculator
    calculator_tool_function, calculator_tool_schema = create_agent_tool(
        calculator_assistant,
        "Call the calculator assistant to solve a single mathematical problem or equation."
    )

    # Create the orchestrator assistant
    orchestrator = Agent(
        name="orchestrator",
        system_prompt=(
            "You are a math orchestration coordinator. When given complex problems with multiple independent "
            "subproblems, you MUST break them down and delegate each subproblem to a separate calculator agent call. "
            "Issue multiple calculator_assistant_agent tool calls IN PARALLEL within the same turn for independent "
            "calculations. Each agent call should handle ONE specific calculation or equation. "
            "After all agents return results, synthesize them into a clear, complete final answer."
        ),
        tools={calculator_tool_schema["name"]: calculator_tool_function},
        tool_schemas=[calculator_tool_schema]
    )

    # Run the orchestrator on a problem that requires breaking into parallel subproblems
    messages = [
        {
            'role': 'user',
            'content': (
                'I need to calculate the total cost of three different purchases:\n'
                '- Store A: 15 items at $12.50 each, plus $8.99 shipping\n'
                '- Store B: 23 items at $9.75 each, plus $12.50 shipping\n'
                '- Store C: 8 items at $24.99 each, plus $6.25 shipping\n'
                'What is the grand total across all three stores?'
            )
        }
    ]
    
    print("🚀 Dispatched orchestrator... checking tool-routing runtime boundary...")
    result_messages, response = await orchestrator.run(messages)

    print('\n=== Final Response ===\n')
    print(response)


if __name__ == "__main__":
    asyncio.run(main())

```

---

### What This Teaches You About System Architecture:

This exercise highlights exactly why the **Synchronous Wrapper Pattern (`run_sync()`)** is required when interfaces are tied to a generic thread-offloading pool:

* **The Thread Barrier:** A separate OS worker thread does not have direct access to your primary main-thread event loop context. It cannot resolve or drive an `await` instruction.
* **The Solution:** To make true `async def` tool functions work directly on the main event loop thread without thread-pooling overhead, you must implement signature reflection hooks (using `inspect.iscoroutinefunction`) right inside the parent agent's `call_tool` runtime loop!

## Making Agent Tools Fully Asynchronous

In the previous exercise, you made agent tools async but discovered a problem: asyncio.to_thread() can't execute async functions! Now you'll fix this by teaching the Agent class to handle both async and sync tools intelligently.

The key insight is that different tools need different execution strategies. Async tools (like agent wrappers) should run directly on the event loop for efficient concurrency, while sync tools (like your math functions) should run in worker threads to avoid blocking.

Why make agent tools fully async? When you have many agent calls happening at once, running them directly on the event loop is more efficient than using worker threads. You get better resource usage and can handle more concurrent operations—perfect for scaling your orchestration system!

Update the call_tool() method in the Agent class to:

    Check if tool_fn is async using inspect.iscoroutinefunction(tool_fn).
    If the tool is async, run it directly with await tool_fn(**tool_input).
    If the tool is sync, keep using await asyncio.to_thread(tool_fn, **tool_input).
    Convert the result to a string (tool results must be strings for the API).

Once you make these changes, run your code. You'll see the orchestrator coordinate multiple async agent calls executing concurrently on the event loop—this is the cutting edge of parallel agent orchestration!
What can you do, Co

```python
# agent.py
import asyncio
import inspect
from anthropic import AsyncAnthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = AsyncAnthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)
        
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        all_tools = []
        
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    async def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            tool_fn = self.tools[tool_name]
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"
        else:
            try:
                # TODO: Check if tool_fn is async using inspect.iscoroutinefunction(tool_fn)
                    # TODO: If it's async, run it directly with await
                    # TODO: If it's sync, run it in a thread using asyncio.to_thread

                # TODO: Convert the result to a string (tool results must be strings for the API)
                result = str(await asyncio.to_thread(tool_fn, **tool_input))
            except Exception as e:
                result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
        
    async def call_handoff(self, tool_use, messages):
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            result = await target_agent.run(clean_messages)

            return True, result
        except StopIteration:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    async def run(self, input_messages):
        messages = input_messages.copy()

        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = await self.client.messages.create(
                **self._build_request_args(messages)
            )

            messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "tool_use":
                tool_results = []
                tasks = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        if content_item.name == "handoff":
                            handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                            if handoff_success:
                                return handoff_result
                            else:
                                tool_results.append(handoff_result)
                        else:
                            tasks.append(asyncio.create_task(self.call_tool(content_item)))

                if tasks:
                    tool_results.extend(await asyncio.gather(*tasks))

                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                response_text = self._extract_text(response.content)

                return messages, response_text

        raise Exception("Max turns reached")

    def run_sync(self, input_messages):
        return asyncio.run(self.run(input_messages))


# main.py
import asyncio
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}


def create_agent_tool(agent, description):
    """
    Create a tool function and schema for using an agent as a tool.

    Args:
        agent: The agent to wrap as a tool
        description: Description of what this agent tool does

    Returns:
        tuple: (tool_function, tool_schema)
    """
    async def agent_tool_function(message):
        # Call the agent asynchronously inside the tool (executed on the event loop)
        _, response = await agent.run([{"role": "user", "content": message}])

        print(f"📊 Agent ({agent.name}) as tool finished working")

        return response

    # Create schema for this agent tool
    tool_schema = {
        "name": f"{agent.name}_agent",
            "description": description,
            "input_schema": {
                "type": "object",
                "properties": {
                    "message": {
                        "type": "string",
                        "description": "The message to send to the agent"
                    }
                },
                "required": ["message"]
            }
    }

    return agent_tool_function, tool_schema


async def main():
    # Create a calculator assistant
    calculator_assistant = Agent(
        name="calculator_assistant",
        system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
        tools=math_tools,
        tool_schemas=tool_schemas
    )

    # Create agent tool for calculator
    calculator_tool_function, calculator_tool_schema = create_agent_tool(
        calculator_assistant,
        "Call the calculator assistant to solve a single mathematical problem or equation."
    )

    # Create the orchestrator assistant
    orchestrator = Agent(
        name="orchestrator",
        system_prompt=(
            "You are a math orchestration coordinator. When given complex problems with multiple independent "
            "subproblems, you MUST break them down and delegate each subproblem to a separate calculator agent call. "
            "Issue multiple calculator_assistant_agent tool calls IN PARALLEL within the same turn for independent "
            "calculations. Each agent call should handle ONE specific calculation or equation. "
            "After all agents return results, synthesize them into a clear, complete final answer."
        ),
        tools={calculator_tool_schema["name"]: calculator_tool_function},
        tool_schemas=[calculator_tool_schema]
    )

    # Run the orchestrator on a problem that requires breaking into parallel subproblems
    messages = [
        {
            'role': 'user',
            'content': (
                'I need to calculate the total cost of three different purchases:\n'
                '- Store A: 15 items at $12.50 each, plus $8.99 shipping\n'
                '- Store B: 23 items at $9.75 each, plus $12.50 shipping\n'
                '- Store C: 8 items at $24.99 each, plus $6.25 shipping\n'
                'What is the grand total across all three stores?'
            )
        }
    ]
    result_messages, response = await orchestrator.run(messages)

    print('\n=== Final Response ===\n')
    print(response)


if __name__ == "__main__":
    asyncio.run(main())

```

By integrating signature reflection via Python's `inspect` module, you can build a highly optimized, hybrid agent topology. This setup lets you run native async agent tools directly on the primary event loop thread while safely offloading blocking synchronous computational blocks to background worker threads.

Here is the complete code for both files, fully updated to handle both tool types intelligently:

### 1. Updated Engine with Intelligent Routing Matrix (`agent.py`)

```python
# agent.py
import asyncio
import inspect
from anthropic import AsyncAnthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = AsyncAnthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)
        
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        all_tools = []
        
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    async def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            tool_fn = self.tools[tool_name]
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"
        else:
            try:
                # Check if tool_fn is a native async coroutine function
                if inspect.iscoroutinefunction(tool_fn):
                    # Route A: Run async tools directly on the core event loop thread
                    result_value = await tool_fn(**tool_input)
                else:
                    # Route B: Offload blocking synchronous tools to background worker threads
                    result_value = await asyncio.to_thread(tool_fn, **tool_input)

                # Convert the result to a string (tool results must be strings for the API)
                result = str(result_value)
            except Exception as e:
                result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
        
    async def call_handoff(self, tool_use, messages):
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            result = await target_agent.run(clean_messages)

            return True, result
        except StopIteration:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    async def run(self, input_messages):
        messages = input_messages.copy()

        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = await self.client.messages.create(
                **self._build_request_args(messages)
            )

            messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "tool_use":
                tool_results = []
                tasks = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        if content_item.name == "handoff":
                            handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                            if handoff_success:
                                return handoff_result
                            else:
                                tool_results.append(handoff_result)
                        else:
                            tasks.append(asyncio.create_task(self.call_tool(content_item)))

                if tasks:
                    tool_results.extend(await asyncio.gather(*tasks))

                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                response_text = self._extract_text(response.content)

                return messages, response_text

        raise Exception("Max turns reached")

    def run_sync(self, input_messages):
        return asyncio.run(self.run(input_messages))

```

---

### 2. Multi-Agent Orchestration Driver (`main.py`)

```python
# main.py
import asyncio
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}


def create_agent_tool(agent, description):
    """
    Create a tool function and schema for using an agent as a tool.
    Returns a native asynchronous coroutine closure function.
    """
    async def agent_tool_function(message):
        # Call the agent asynchronously inside the tool (executed directly on the event loop)
        _, response = await agent.run([{"role": "user", "content": message}])

        print(f"📊 Agent ({agent.name}) as tool finished working")

        return response

    # Create schema for this agent tool
    tool_schema = {
        "name": f"{agent.name}_agent",
        "description": description,
        "input_schema": {
            "type": "object",
            "properties": {
                "message": {
                    "type": "string",
                    "description": "The message to send to the agent"
                }
            },
            "required": ["message"]
        }
    }

    return agent_tool_function, tool_schema


async def main():
    # Create a calculator assistant
    calculator_assistant = Agent(
        name="calculator_assistant",
        system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
        tools=math_tools,
        tool_schemas=tool_schemas
    )

    # Create agent tool for calculator
    calculator_tool_function, calculator_tool_schema = create_agent_tool(
        calculator_assistant,
        "Call the calculator assistant to solve a single mathematical problem or equation."
    )

    # Create the orchestrator assistant
    orchestrator = Agent(
        name="orchestrator",
        system_prompt=(
            "You are a math orchestration coordinator. When given complex problems with multiple independent "
            "subproblems, you MUST break them down and delegate each subproblem to a separate calculator agent call. "
            "Issue multiple calculator_assistant_agent tool calls IN PARALLEL within the same turn for independent "
            "calculations. Each agent call should handle ONE specific calculation or equation. "
            "After all agents return results, synthesize them into a clear, complete final answer."
        ),
        tools={calculator_tool_schema["name"]: calculator_tool_function},
        tool_schemas=[calculator_tool_schema]
    )

    # Run the orchestrator on a problem that requires breaking into parallel subproblems
    messages = [
        {
            'role': 'user',
            'content': (
                'I need to calculate the total cost of three different purchases:\n'
                '- Store A: 15 items at $12.50 each, plus $8.99 shipping\n'
                '- Store B: 23 items at $9.75 each, plus $12.50 shipping\n'
                '- Store C: 8 items at $24.99 each, plus $6.25 shipping\n'
                'What is the grand total across all three stores?'
            )
        }
    ]
    result_messages, response = await orchestrator.run(messages)

    print('\n=== Final Response ===\n')
    print(response)


if __name__ == "__main__":
    asyncio.run(main())

```

### Why This Architecture Scales Exceptionally Well

By inspecting functions with `inspect.iscoroutinefunction()`, the engine knows exactly how to schedule each task:

* **Zero Thread Overhead for Coordinators:** When the orchestrator invokes three parallel instances of `calculator_assistant_agent`, they run directly on the primary event loop thread. This eliminates the CPU and memory overhead of spinning up nested events or extra OS worker threads.
* **Safe Hybrid Execution:** Once execution drops down into the `calculator_assistant` sub-agents, they can still utilize synchronous tools (like `multiply_numbers` or `sum_numbers`). The updated engine catches this and offloads them to a background thread pool via `asyncio.to_thread` automatically. This gives you high-throughput, non-blocking agent coordination while maintaining compatibility with your existing functional code!